# VaxFlow — Monthly Retraining Notebook

Run this notebook **once a month** after the month ends.

**Steps:**
1. Fill in Cell 2 with the actual data for the month that just ended
2. Run all cells (Kernel → Restart & Run All)
3. Copy the new `.pkl` and `.json` files into your `fast_api/` folder
4. Restart the FastAPI server

That's it. The model will retrain on the updated data and predictions will improve.

In [1]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    r2_score, mean_absolute_percentage_error
)

print('Libraries loaded.')

Libraries loaded.


In [2]:
# ── Cell 2: FILL THIS IN EVERY MONTH ─────────────────────────────────────────
# Enter the actual data for the month that just ended.
# Get these numbers from your ABTC monthly report.
#
# Example: if May 2026 just ended, fill in May 2026's actual numbers.

NEW_ROW = {
    # ── Time ──────────────────────────────────────────────────────────────────
    'year':  2026,          # ← change this
    'month': 6,             # ← change this (1=Jan, 6=Jun, 12=Dec)

    # ── Climate (from PAGASA monthly report for CDO) ───────────────────────────
    'temperature_c':    29.2,   # average temperature this month
    'rainfall_mm':      145.0,  # total rainfall this month
    'humidity_percent': 80.5,   # average humidity
    'heat_index_c':     33.1,   # average heat index

    # ── Animal / Population ───────────────────────────────────────────────────
    'dog_population':           194000,  # estimated dog population CDO
    'dog_population_growth_rate': 0.003, # annual growth rate (decimal)
    'urban_density':            4580.0,  # people per sq km
    'stray_density_index':      6.9,     # 1-10 scale
    'breeding_season_cycle':    1,       # 1 if breeding season, 0 if not

    # ── Epidemiology (from ABTC monthly register) ─────────────────────────────
    'bite_cases_total':      340,    # total animal bite patients this month
    'category_1_cases':       65,    # Category I (licks on intact skin)
    'category_2_cases':      140,    # Category II (minor scratches)
    'category_3_cases':      135,    # Category III (transdermal bites)
    'rabies_confirmation_rate': 0.03, # confirmed rabies / total bites

    # ── Vaccine Operations ────────────────────────────────────────────────────
    'arv_doses_administered': 1712,  # ← MOST IMPORTANT: actual ARV doses given
    'rig_vials_used':          120,  # RIG vials used this month
    'vaccination_coverage_rate': 0.70, # dog vaccination coverage rate
    'pep_completion_rate':     0.82,   # % of patients who completed PEP
    'rig_availability_rate':   0.93,   # % of time RIG was available
    'cold_chain_capacity_index': 0.90, # cold chain performance (0-1)
    'procurement_delay_days':    0,    # days of procurement delay this month
    'stockout_flag':             0,    # 1 if stockout occurred, 0 if not

    # ── Environment / Socioeconomic ───────────────────────────────────────────
    'waste_management_index':     6.2, # 1-10 scale (higher = better)
    'poverty_index':             16.1, # poverty rate %
    'travel_accessibility_index': 6.3, # 1-10 scale
    'public_awareness_score':     7.5, # 1-10 scale

    # ── Flags (1 = yes, 0 = no) ───────────────────────────────────────────────
    'dog_vaccination_campaign_flag': 0, # was there a dog vacc campaign?
    'extreme_weather_flag':          0, # typhoon / extreme weather event?
    'holiday_season_flag':           0, # major holiday season?
    'school_vacation_flag':          1, # school vacation month?
}

print(f"New row ready: {NEW_ROW['year']}-{NEW_ROW['month']:02d}  "
      f"ARV doses = {NEW_ROW['arv_doses_administered']}")

New row ready: 2026-06  ARV doses = 1712


In [3]:
# ── Cell 3: Load existing dataset ─────────────────────────────────────────────
DATASET_PATH = 'vaxflow_synthesized_dataset_final.xlsx'

df = pd.read_excel(DATASET_PATH)
df = df.sort_values(['year', 'month']).reset_index(drop=True)

print(f'Loaded dataset: {df.shape[0]} rows  ({df.year.min()} – {df.year.max()})')

# Check if this month already exists
exists = df[(df['year'] == NEW_ROW['year']) & (df['month'] == NEW_ROW['month'])]
if len(exists) > 0:
    print(f"⚠️  Row for {NEW_ROW['year']}-{NEW_ROW['month']:02d} already exists — will UPDATE it with new values.")
    df = df[~((df['year'] == NEW_ROW['year']) & (df['month'] == NEW_ROW['month']))]
else:
    print(f"✅ New row for {NEW_ROW['year']}-{NEW_ROW['month']:02d} — will APPEND to dataset.")

Loaded dataset: 228 rows  (2010 – 2028)
⚠️  Row for 2026-06 already exists — will UPDATE it with new values.


In [4]:
# ── Cell 4: Append new row and save updated dataset ───────────────────────────
new_df = pd.DataFrame([NEW_ROW])
df = pd.concat([df, new_df], ignore_index=True)
df = df.sort_values(['year', 'month']).reset_index(drop=True)

# Save updated dataset
df.to_excel(DATASET_PATH, index=False)

print(f'Dataset updated: now {df.shape[0]} rows  ({df.year.min()} – {df.year.max()})')
print(f'Last 3 rows:')
print(df[['year','month','arv_doses_administered']].tail(3).to_string(index=False))

Dataset updated: now 228 rows  (2010 – 2028)
Last 3 rows:
 year  month  arv_doses_administered
 2028     10                    1865
 2028     11                    1733
 2028     12                    1678


In [5]:
# ── Cell 5: Feature engineering (identical to original notebook) ──────────────
df_fe = df.copy()

# Lag features
for lag in [1, 2, 3, 6, 12]:
    df_fe[f'arv_lag_{lag}'] = df_fe['arv_doses_administered'].shift(lag)

# Rolling averages (shifted to avoid leakage)
df_fe['arv_roll_3'] = df_fe['arv_doses_administered'].shift(1).rolling(3).mean()
df_fe['arv_roll_6'] = df_fe['arv_doses_administered'].shift(1).rolling(6).mean()

# Cyclical month encoding
df_fe['month_sin'] = np.sin(2 * np.pi * df_fe['month'] / 12)
df_fe['month_cos'] = np.cos(2 * np.pi * df_fe['month'] / 12)

# Interaction features
df_fe['temp_x_breeding'] = df_fe['temperature_c'] * df_fe['breeding_season_cycle']
df_fe['stray_x_waste']   = df_fe['stray_density_index'] * (10 - df_fe['waste_management_index'])
df_fe['bite_per_dog']    = df_fe['bite_cases_total'] / (df_fe['dog_population'] / 1000)

# Year trend
df_fe['year_trend'] = df_fe['year'] - df_fe['year'].min()

# Drop leaky column and rows with NaN from lag
df_fe = df_fe.drop(columns=['rig_vials_used'], errors='ignore')
df_fe = df_fe.dropna().reset_index(drop=True)

print(f'Feature engineering done: {df_fe.shape[0]} rows, {df_fe.shape[1]} columns')

Feature engineering done: 216 rows, 44 columns


In [6]:
# ── Cell 6: Prepare features and target ───────────────────────────────────────
TARGET = 'arv_doses_administered'

DROP_COLS = [TARGET, 'date'] if 'date' in df_fe.columns else [TARGET]
feature_cols = [c for c in df_fe.columns if c not in DROP_COLS]

X = df_fe[feature_cols]
y = df_fe[TARGET]

# Chronological 70/15/15 split
n = len(df_fe)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train, y_train = X.iloc[:train_end],  y.iloc[:train_end]
X_val,   y_val   = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test,  y_test  = X.iloc[val_end:],    y.iloc[val_end:]

print(f'Features: {len(feature_cols)}')
print(f'Train: {len(X_train)} rows  |  Val: {len(X_val)} rows  |  Test: {len(X_test)} rows')

Features: 43
Train: 151 rows  |  Val: 32 rows  |  Test: 33 rows


In [7]:
# ── Cell 7: Retrain the model ─────────────────────────────────────────────────
model = Ridge(alpha=1.0, fit_intercept=True)
model.fit(X_train, y_train)

# Evaluate on all splits
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    print(f'{label:10s}  R²={r2:.4f}  MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.2f}%')
    return {'R2': round(r2,4), 'MAE': round(mae,2), 'RMSE': round(rmse,2), 'MAPE_pct': round(mape,2)}

train_m = metrics(y_train, model.predict(X_train), 'Train')
val_m   = metrics(y_val,   model.predict(X_val),   'Validation')
test_m  = metrics(y_test,  model.predict(X_test),  'Test')

gap = train_m['R2'] - test_m['R2']
print(f'\nTrain-Test gap: {gap:.4f}  {"✅ Healthy" if gap < 0.10 else "⚠️ Check for overfitting"}')

Train       R²=0.9749  MAE=23.34  RMSE=30.17  MAPE=1.41%
Validation  R²=0.9251  MAE=34.01  RMSE=42.68  MAPE=1.85%
Test        R²=0.9136  MAE=33.91  RMSE=45.76  MAPE=1.86%

Train-Test gap: 0.0613  ✅ Healthy


In [8]:
# ── Cell 8: Compare new metrics vs previous model ─────────────────────────────
PKL_PATH = 'vaxflow_arv_model.pkl'

try:
    old_payload = joblib.load(PKL_PATH)
    old_m = old_payload.get('test_metrics', {})
    print('=== COMPARISON: Previous vs New Model ===')
    print(f"{'Metric':<10} {'Previous':>12} {'New':>12} {'Change':>12}")
    print('-' * 48)
    for key, label in [('R2','R²'), ('MAE','MAE'), ('RMSE','RMSE'), ('MAPE_pct','MAPE %')]:
        old_val = old_m.get(key, 0)
        new_val = test_m.get(key, 0)
        # For R², higher is better. For MAE/RMSE/MAPE, lower is better.
        if key == 'R2':
            delta = new_val - float(old_val)
            arrow = '✅ improved' if delta > 0 else ('⚠️ worse' if delta < -0.01 else '➡️ same')
        else:
            delta = float(old_val) - new_val
            arrow = '✅ improved' if delta > 0 else ('⚠️ worse' if delta < -1 else '➡️ same')
        print(f"{label:<10} {str(old_val):>12} {str(new_val):>12}  {arrow}")
except FileNotFoundError:
    print('No previous model found — this is the first training run.')

=== COMPARISON: Previous vs New Model ===
Metric         Previous          New       Change
------------------------------------------------
R²               0.9069       0.9136  ✅ improved
MAE               32.22        33.91  ⚠️ worse
RMSE              44.76        45.76  ➡️ same
MAPE %             1.74         1.86  ➡️ same


In [9]:
# ── Cell 9: Save new model .pkl ───────────────────────────────────────────────
payload = {
    'model':        model,
    'model_name':   'Linear Regression',
    'feature_cols': feature_cols,
    'split_ratio':  '70:15:15 chronological',
    'test_metrics': test_m,
    'trained_on': {
        'years':   f"{int(df_fe['year'].min())}–{int(df_fe['year'].max())}",
        'samples': len(df_fe),
    }
}

joblib.dump(payload, PKL_PATH)
print(f'✅ Saved: {PKL_PATH}')
print(f'   Trained on {len(df_fe)} rows  ({payload["trained_on"]["years"]})')
print(f'   Test R² = {test_m["R2"]}  |  MAPE = {test_m["MAPE_pct"]}%')

✅ Saved: vaxflow_arv_model.pkl
   Trained on 216 rows  (2011–2028)
   Test R² = 0.9136  |  MAPE = 1.86%


In [10]:
# ── Cell 10: Regenerate forecast JSON ─────────────────────────────────────────
# Recompute predictions for all rows and save the forecast JSON
# that the frontend uses for the ARV Demand Forecast tab.

MONTH_NAMES = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

n_total  = len(df_fe)
train_end_new = int(n_total * 0.70)
val_end_new   = int(n_total * 0.85)

def get_split(idx):
    if idx < train_end_new: return 'train'
    if idx < val_end_new:   return 'validation'
    return 'test'

y_all_pred = model.predict(X)

records = []
for i, (_, row) in enumerate(df_fe.iterrows()):
    pred  = max(0, round(float(y_all_pred[i])))
    actual = int(row[TARGET])
    records.append({
        'year':        int(row['year']),
        'month':       int(row['month']),
        'monthName':   MONTH_NAMES[int(row['month']) - 1],
        'predicted':   pred,
        'actual':      actual,
        'recommended': round(pred * 1.12),
        'split':       get_split(i),
    })

JSON_PATH = 'vaxflow_forecast_monthly.json'
with open(JSON_PATH, 'w') as f:
    json.dump(records, f, indent=2)

print(f'✅ Saved: {JSON_PATH}  ({len(records)} records)')
print(f'   Years: {records[0]["year"]} – {records[-1]["year"]}')

✅ Saved: vaxflow_forecast_monthly.json  (216 records)
   Years: 2011 – 2028


In [11]:
# ── Cell 11: Verify the new prediction for next month ─────────────────────────
# Shows what the retrained model now predicts for the NEXT month.
# This is what the Live Predict tab will return after you deploy.

next_month = NEW_ROW['month'] % 12 + 1
next_year  = NEW_ROW['year'] + (1 if NEW_ROW['month'] == 12 else 0)

# Find the base row for next month
df_sorted = df_fe.sort_values(['year','month']).reset_index(drop=True)
base      = df_sorted[df_sorted['month'] == next_month].iloc[-1].copy()

def get_lag(n):
    idx = df_sorted[(df_sorted['year'] == base['year']) &
                    (df_sorted['month'] == next_month)].index
    if len(idx) > 0 and idx[0] - n >= 0:
        return float(df_sorted.iloc[idx[0] - n][TARGET])
    return float(df_sorted[df_sorted['month'] == next_month][TARGET].mean())

feat_dict = {c: float(base[c]) for c in feature_cols if c in base}
feat_dict['year']       = next_year
feat_dict['month']      = next_month
feat_dict['arv_lag_1']  = get_lag(1)
feat_dict['arv_lag_2']  = get_lag(2)
feat_dict['arv_lag_3']  = get_lag(3)
feat_dict['arv_lag_6']  = get_lag(6)
feat_dict['arv_lag_12'] = get_lag(12)
feat_dict['arv_roll_3'] = float(np.mean([get_lag(1), get_lag(2), get_lag(3)]))
feat_dict['arv_roll_6'] = float(np.mean([get_lag(i) for i in range(1,7)]))
feat_dict['month_sin']  = np.sin(2 * np.pi * next_month / 12)
feat_dict['month_cos']  = np.cos(2 * np.pi * next_month / 12)
feat_dict['year_trend'] = next_year - int(df['year'].min())

import pandas as pd
X_next = pd.DataFrame([[feat_dict.get(c, 0) for c in feature_cols]], columns=feature_cols)
pred_next = max(0, round(float(model.predict(X_next)[0])))

print(f'\n=== NEXT MONTH PREDICTION ===')
print(f'Month:             {MONTH_NAMES[next_month-1]} {next_year}')
print(f'Predicted doses:   {pred_next:,}')
print(f'Recommended order: {round(pred_next * 1.12):,}  (incl. 12% buffer)')
print(f'\nThis is what the Live Predict tab will return after deployment.')


=== NEXT MONTH PREDICTION ===
Month:             July 2026
Predicted doses:   1,692
Recommended order: 1,895  (incl. 12% buffer)

This is what the Live Predict tab will return after deployment.


In [12]:
# ── Cell 12: Deployment checklist ─────────────────────────────────────────────
import os

print('=== DEPLOYMENT CHECKLIST ===')
print()

files = {
    'vaxflow_arv_model.pkl':              'Trained model — copy to fast_api/',
    'vaxflow_forecast_monthly.json':      'Forecast data — copy to fast_api/',
    'vaxflow_synthesized_dataset_final.xlsx': 'Updated dataset — copy to fast_api/',
}

all_good = True
for fname, desc in files.items():
    exists = os.path.exists(fname)
    status = '✅' if exists else '❌ MISSING'
    print(f'  {status}  {fname}')
    print(f'         → {desc}')
    if not exists:
        all_good = False

print()
if all_good:
    print('All files ready.')
    print()
    print('Next steps:')
    print('  1. Copy all 3 files above into your fast_api/ folder')
    print('  2. Restart the FastAPI server (Ctrl+C then uvicorn fast_api.main:app --reload)')
    print('  3. Open the dashboard — the ML banner will show updated predictions')
    print(f'  4. In Live Predict, try {MONTH_NAMES[next_month-1]} {next_year}')
    print(f'     — should return {pred_next:,} predicted doses')
else:
    print('⚠️  Some files are missing. Re-run the notebook from Cell 1.')

=== DEPLOYMENT CHECKLIST ===

  ✅  vaxflow_arv_model.pkl
         → Trained model — copy to fast_api/
  ✅  vaxflow_forecast_monthly.json
         → Forecast data — copy to fast_api/
  ✅  vaxflow_synthesized_dataset_final.xlsx
         → Updated dataset — copy to fast_api/

All files ready.

Next steps:
  1. Copy all 3 files above into your fast_api/ folder
  2. Restart the FastAPI server (Ctrl+C then uvicorn fast_api.main:app --reload)
  3. Open the dashboard — the ML banner will show updated predictions
  4. In Live Predict, try July 2026
     — should return 1,692 predicted doses
